# Lesson 04 Lab — Why BF16 Is Often the First Low-Precision Choice

**Puzzle:** FP16 and BF16 both use 16 bits. Why can their numerical behavior differ dramatically?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

FP16 and BF16 both occupy 16 bits, but FP16 uses 5 exponent and 10 fraction bits whereas BF16 uses 8 exponent and 7 fraction bits. The former offers finer local spacing; the latter offers a much larger dynamic range.

### Core mechanism

Rounding error is governed by representable spacing near a value, while overflow is governed by exponent range. Accumulation policy adds a third variable: low-precision inputs may still accumulate into a wider type depending on the operator.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "04-bf16-first"
device = require_cuda()
torch.manual_seed(2026 + 4)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

BF16 can avoid FP16 overflow but may show larger rounding error on well-scaled values. FP32 is a useful numerical reference, not automatically the production throughput winner.

### What this code tests

The lab separates large-value representability, GEMM error, and GEMM latency into three observations so one does not stand in for the others.

**Experiment:** Compare range, matrix-multiplication error, and CUDA timing for FP32, FP16, and BF16.

**Declared evidence label:** `pytorch-gpu`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
n = 1536; a32 = torch.randn(n, n, device=device); b32 = torch.randn(n, n, device=device)
ref = a32 @ b32; rows = {}
for dtype in (torch.float32, torch.float16, torch.bfloat16):
    a, b = a32.to(dtype), b32.to(dtype); out = a @ b
    rows[str(dtype).split(".")[-1]] = {"timing": cuda_benchmark(lambda: a @ b, warmup=4, repeats=12),
                                       "error": error_metrics(ref, out.float())}
range_probe = {"fp16_1e5_finite": bool(torch.isfinite(torch.tensor([1e5], device=device).half()).item()),
               "bf16_1e5_finite": bool(torch.isfinite(torch.tensor([1e5], device=device).bfloat16()).item()),
               "fp16_max": torch.finfo(torch.float16).max, "bf16_max": torch.finfo(torch.bfloat16).max}
result = base_result(4, "pytorch-gpu"); result.update({"matrix_shape": [n, n], "formats": rows,
    "range_probe": range_probe, "conclusion": "BF16 preserved the large-value range while FP16 and BF16 showed different accuracy/performance trade-offs."})


## 3. Inspect the evidence

Look separately at overflow behavior, error against FP32, and latency. No single column decides every workload.

### Acceptance and rollback gate

Test both a range probe and workload output error against FP32, then measure latency on the target shape. Keep BF16 only when stability and performance meet the frozen thresholds.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "BF16 preserved the large-value range while FP16 and BF16 showed different accuracy/performance trade-offs.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:45:14+00:00",
  "formats": {
    "bfloat16": {
      "error": {
        "cosine": 0.99999589,
        "mae": 0.08880086,
        "max_abs": 0.71353149,
        "rmse": 0.11277169
      },
      "timing": {
        "median_ms": 0.04416,
        "p90_ms": 0.044704,
        "repeats": 12,
        "samples_ms": [
          0.045888,
          0.044128,
          0.043488,
          0.0432,
          0.043456,
          0.044192,
          0.044064,
          0.044928,
          0.04432,
          0.044416,
          0.044704,
          0.043872
        ],
        "warmup": 4
      }
    },
    "fl

## 4. Explain the result

BF16 is a pragmatic stability-first baseline on supported hardware, but workload-specific error and speed still need measurement.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).